In [ ]:
# In[1]:

import pandas as pd
import pytz

# load CSVs (they are in the current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# helper to convert epoch seconds to Asia/Shanghai tz string
tz = pytz.timezone('Asia/Shanghai')
def ts_range_series(s):
    if s.empty:
        return (None, None)
    mn = pd.to_datetime(int(s.min()), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    mx = pd.to_datetime(int(s.max()), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    return mn, mx

# helper to get top-n unique values by frequency (compact)
def top_unique_series(s, n):
    if s is None or s.size == 0:
        return []
    top = s.value_counts().index[:n].tolist()
    return top

# compute aggregates for metric.csv
m_count = len(metric_df)
m_min_ts, m_max_ts = ts_range_series(metric_df['timestamp'])
m_unique_cmdb = top_unique_series(metric_df['cmdb_id'].dropna(), 20)
m_unique_kpi = top_unique_series(metric_df['kpi_name'].dropna(), 50)
m_distinct_pairs = metric_df[['cmdb_id','kpi_name']].dropna().drop_duplicates().shape[0]

# compute aggregates for trace.csv
t_count = len(trace_df)
t_min_ts, t_max_ts = ts_range_series(trace_df['timestamp'])
t_unique_cmdb = top_unique_series(trace_df['cmdb_id'].dropna(), 20)
t_unique_trace = top_unique_series(trace_df['trace_name'].dropna(), 50)

# compute aggregates for log.csv
l_count = len(log_df)
l_min_ts, l_max_ts = ts_range_series(log_df['timestamp'])
l_unique_cmdb = top_unique_series(log_df['cmdb_id'].dropna(), 20)
l_unique_log = top_unique_series(log_df['log_name'].dropna(), 50)

# compute aggregates for error_logs.csv
e_count = len(error_logs_df)
e_min_ts, e_max_ts = ts_range_series(error_logs_df['timestamp'])
e_unique_cmdb = top_unique_series(error_logs_df['cmdb_id'].dropna(), 20)
# error_logs has 'message' column; present a top list of messages (compact)
e_unique_message = top_unique_series(error_logs_df['message'].dropna(), 50)

# Build a compact summary DataFrame (one row per file). Keep lists truncated for readability.
def shorten_list(lst, limit):
    if not lst:
        return ""
    if len(lst) <= limit:
        return ", ".join(map(str, lst))
    return ", ".join(map(str, lst[:limit])) + f", ... (+{len(lst)-limit} more)"

summary_df = pd.DataFrame([
    {
        "file": "metric.csv",
        "rows": m_count,
        "min_timestamp_ASIA/Shanghai": m_min_ts,
        "max_timestamp_ASIA/Shanghai": m_max_ts,
        "unique_cmdb_count": metric_df['cmdb_id'].nunique(dropna=True),
        "unique_cmdb_top20": shorten_list(m_unique_cmdb, 20),
        "unique_kpi_count": metric_df['kpi_name'].nunique(dropna=True),
        "unique_kpi_top50": shorten_list(m_unique_kpi, 10),  # show at most 10 in table for compactness
        "distinct_cmdb_kpi_pairs": m_distinct_pairs
    },
    {
        "file": "trace.csv",
        "rows": t_count,
        "min_timestamp_ASIA/Shanghai": t_min_ts,
        "max_timestamp_ASIA/Shanghai": t_max_ts,
        "unique_cmdb_count": trace_df['cmdb_id'].nunique(dropna=True),
        "unique_cmdb_top20": shorten_list(t_unique_cmdb, 20),
        "unique_trace_count": trace_df['trace_name'].nunique(dropna=True),
        "unique_trace_top50": shorten_list(t_unique_trace, 10),
        "distinct_cmdb_kpi_pairs": ""
    },
    {
        "file": "log.csv",
        "rows": l_count,
        "min_timestamp_ASIA/Shanghai": l_min_ts,
        "max_timestamp_ASIA/Shanghai": l_max_ts,
        "unique_cmdb_count": log_df['cmdb_id'].nunique(dropna=True),
        "unique_cmdb_top20": shorten_list(l_unique_cmdb, 20),
        "unique_log_count": log_df['log_name'].nunique(dropna=True),
        "unique_log_top50": shorten_list(l_unique_log, 10),
        "distinct_cmdb_kpi_pairs": ""
    },
    {
        "file": "error_logs.csv",
        "rows": e_count,
        "min_timestamp_ASIA/Shanghai": e_min_ts,
        "max_timestamp_ASIA/Shanghai": e_max_ts,
        "unique_cmdb_count": error_logs_df['cmdb_id'].nunique(dropna=True),
        "unique_cmdb_top20": shorten_list(e_unique_cmdb, 20),
        "unique_message_count": error_logs_df['message'].nunique(dropna=True),
        "unique_message_top50": shorten_list(e_unique_message, 10),
        "distinct_cmdb_kpi_pairs": ""
    }
])

# Keep display compact: select columns to show and limit rows (we have 4 rows only)
summary_df[[
    "file","rows","min_timestamp_ASIA/Shanghai","max_timestamp_ASIA/Shanghai",
    "unique_cmdb_count","unique_cmdb_top20",
    # these next columns vary by file; some will be NaN/empty
    "unique_kpi_count","unique_kpi_top50",
    "unique_trace_count","unique_trace_top50",
    "unique_log_count","unique_log_top50",
    "unique_message_count","unique_message_top50",
    "distinct_cmdb_kpi_pairs"
]].head(20)

```
Out[1]:
```
```python
# Build a plain-English summary string from the previously computed summary_df
row = summary_df.set_index('file')

m = row.loc['metric.csv']
t = row.loc['trace.csv']
l = row.loc['log.csv']
e = row.loc['error_logs.csv']

summary = (
    "Summary of telemetry files (timestamps shown in Asia/Shanghai, CST):\n\n"
    f"metric.csv: {int(m['rows']):,} rows; time range {m['min_timestamp_ASIA/Shanghai']} to {m['max_timestamp_ASIA/Shanghai']}; "
    f"{int(m['unique_cmdb_count'])} unique cmdb_id values; {int(m['unique_kpi_count'])} unique KPIs; "
    f"{int(m['distinct_cmdb_kpi_pairs'])} distinct (cmdb_id, kpi_name) pairs.\n\n"
    f"trace.csv: {int(t['rows']):,} rows; time range {t['min_timestamp_ASIA/Shanghai']} to {t['max_timestamp_ASIA/Shanghai']}; "
    f"{int(t['unique_cmdb_count'])} unique cmdb_id values; {int(t['unique_trace_count'])} unique trace names.\n\n"
    f"log.csv: {int(l['rows']):,} rows; time range {l['min_timestamp_ASIA/Shanghai']} to {l['max_timestamp_ASIA/Shanghai']}; "
    f"{int(l['unique_cmdb_count'])} unique cmdb_id values; {int(l['unique_log_count'])} unique log names.\n\n"
    f"error_logs.csv: {int(e['rows']):,} rows; time range {e['min_timestamp_ASIA/Shanghai']} to {e['max_timestamp_ASIA/Shanghai']}; "
    f"{int(e['unique_cmdb_count'])} unique cmdb_id values; {int(e['unique_message_count'])} unique messages."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

file    rows min_timestamp_ASIA/Shanghai max_timestamp_ASIA/Shanghai  unique_cmdb_count                                  unique_cmdb_top20  unique_kpi_count                                   unique_kpi_top50  unique_trace_count                                 unique_trace_top50  unique_log_count                unique_log_top50  unique_message_count                               unique_message_top50  \
0      metric.csv  325563     2022-03-21 15:00:00 CST     2022-03-21 16:00:00 CST                 59  adservice, adservice2, frontend2-0, frontend-2...            1428.0  container.node-6.container_network_transmit_pa...                 NaN                                                NaN               NaN                             NaN                   NaN                                                NaN   
1       trace.csv   32628     2022-03-21 15:30:00 CST     2022-03-21 15:59:00 CST                 40  frontend-1, frontend-2, frontend-0, checkoutse...               NaN                                                NaN               200.0  trace.from_frontend-1.row_count, trace.from_fr...               NaN                             NaN                   NaN                                                NaN   
2         log.csv    1834     2022-03-21 15:30:00 CST     2022-03-21 16:00:00 CST                 31  cartservice-2, cartservice-1, cartservice-0, c...               NaN                                                NaN                 NaN                                                NaN               2.0  log.error_count, log.row_count                   NaN                                                NaN   
3  error_logs.csv     640     2022-03-21 15:45:39 CST     2022-03-21 15:50:03 CST                  6  frontend-1, frontend-2, cartservice-1, cartser...               NaN                                                NaN                 NaN                                                NaN               NaN                             NaN                 199.0  severity: error, message: request error,      ...   

  distinct_cmdb_kpi_pairs  
0                    5350  
1                          
2                          
3```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric_df from previous state
tz = pytz.timezone('Asia/Shanghai')

# 1) compute global P95 per (cmdb_id, kpi_name) using entire metric_df (global threshold)
global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) filter metric_df to incident window in Asia/Shanghai (inclusive)
start_local = pd.Timestamp('2022-03-21 15:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 16:00:00', tz=tz)
# convert to UTC epoch seconds for numeric timestamp comparison
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) join global P95 into window data and mark anomalies
dfw = metric_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
dfw['is_anomaly'] = (dfw['value'] >= dfw['global_P95']).astype(int)

# aggregate per (cmdb_id, kpi_name) present in window
agg = (
    dfw
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        window_count=('value', 'size'),
        anomaly_count=('is_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest anomaly timestamp per group (if any)
earliest_anom = (
    dfw[dfw['is_anomaly'] == 1]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'earliest_anomaly_timestamp_ts'})
)

# merge earliest anomaly ts and global_P95
res = agg.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left').merge(
    earliest_anom, on=['cmdb_id', 'kpi_name'], how='left'
)

# convert earliest_anomaly_timestamp to Asia/Shanghai string, keep nulls as None
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

res['earliest_anomaly_timestamp'] = res['earliest_anomaly_timestamp_ts'].apply(fmt_ts)
res.drop(columns=['earliest_anomaly_timestamp_ts'], inplace=True)

# severity = max_value_in_window - global_P95 (if global_P95 missing, set NaN)
res['max_value_minus_P95'] = res['max_value_in_window'] - res['global_P95']

# keep desired columns and order, round numeric columns for compactness
out_cols = [
    'cmdb_id','kpi_name','global_P95','window_count','anomaly_count',
    'earliest_anomaly_timestamp','max_value_in_window','max_value_minus_P95'
]
res['global_P95'] = res['global_P95'].round(6)
res['max_value_in_window'] = res['max_value_in_window'].round(6)
res['max_value_minus_P95'] = res['max_value_minus_P95'].round(6)

# 4) order by anomaly_count desc, then severity desc, limit top 20
res_sorted = res.sort_values(by=['anomaly_count','max_value_minus_P95'], ascending=[False, False])
top20 = res_sorted[out_cols].head(20).reset_index(drop=True)

# 5) small aggregate: total distinct pairs considered and how many had anomaly_count>0
total_pairs = int(res.shape[0])
pairs_with_anomaly = int((res['anomaly_count'] > 0).sum())
summary = pd.DataFrame([{
    'total_distinct_cmdb_kpi_pairs_in_window': total_pairs,
    'pairs_with_anomaly_count_gt_0': pairs_with_anomaly
}])

# Display compact results: top20 rows and the summary line
top20, summary

```
Out[2]:
```
```python
# Build a plain-English summary string from the existing results (top20 and summary variables)
top = top20
agg = summary.loc[0] if 'summary' in globals() else None

total_pairs = int(agg['total_distinct_cmdb_kpi_pairs_in_window']) if agg is not None else None
pairs_with_anom = int(agg['pairs_with_anomaly_count_gt_0']) if agg is not None else None
percent = (pairs_with_anom / total_pairs * 100) if total_pairs else None

summary_text = (
    "Metric analysis summary (incident window 2022-03-21 15:30:00 to 16:00:00 Asia/Shanghai):\n\n"
    f"- Total distinct (cmdb_id, kpi_name) present in the window: {total_pairs}\n"
    f"- Number with anomaly_count > 0 in the window: {pairs_with_anom} "
    f"({percent:.1f}% of total)\n\n"
    "Top findings (top-ranked by anomaly_count then severity):\n"
    f"1) {top.loc[0,'cmdb_id']} / {top.loc[0,'kpi_name']} -> global_P95={top.loc[0,'global_P95']}, "
    f"window_count={int(top.loc[0,'window_count'])}, anomaly_count={int(top.loc[0,'anomaly_count'])}, "
    f"earliest_anomaly={top.loc[0,'earliest_anomaly_timestamp']}, max_in_window={top.loc[0,'max_value_in_window']}, "
    f"severity={top.loc[0,'max_value_minus_P95']}\n"
    f"2) {top.loc[1,'cmdb_id']} / {top.loc[1,'kpi_name']} -> global_P95={top.loc[1,'global_P95']}, "
    f"window_count={int(top.loc[1,'window_count'])}, anomaly_count={int(top.loc[1,'anomaly_count'])}, "
    f"earliest_anomaly={top.loc[1,'earliest_anomaly_timestamp']}, max_in_window={top.loc[1,'max_value_in_window']}, "
    f"severity={top.loc[1,'max_value_minus_P95']}\n"
    f"3) {top.loc[2,'cmdb_id']} / {top.loc[2,'kpi_name']} -> global_P95={top.loc[2,'global_P95']}, "
    f"window_count={int(top.loc[2,'window_count'])}, anomaly_count={int(top.loc[2,'anomaly_count'])}, "
    f"earliest_anomaly={top.loc[2,'earliest_anomaly_timestamp']}, max_in_window={top.loc[2,'max_value_in_window']}, "
    f"severity={top.loc[2,'max_value_minus_P95']}\n\n"
    "Notes:\n"
    "- Many top-ranked KPIs show anomaly_count = window_count = 31 (every point in the window exceeded its global P95).\n"
    "- Several mesh/istio metrics have global_P95 = 0.0 while showing nonzero maxima in the window, so they appear as anomalies by the P95 threshold.\n"
    "- Returned table contains up to 20 rows (top 20 pairs) ordered by anomaly_count then severity; overall there were "
    f"{total_pairs} pairs considered and {pairs_with_anom} with anomalies.\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  global_P95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  max_value_minus_P95
0          frontend-2              container.node-5.container_memory_rss  18079744.0            31             31    2022-03-21 15:30:00 CST           18147328.0              67584.0
1          frontend-2  mesh.source.frontend.cartservice.istio_request...         0.0            31             31    2022-03-21 15:30:00 CST              13175.0              13175.0
2         frontend2-0  mesh.destination.unknown.frontend2.istio_respo...         0.0            31             31    2022-03-21 15:30:00 CST               9900.0               9900.0
3       cartservice-1  mesh.destination.checkoutservice.cartservice.i...         0.0            31             31    2022-03-21 15:30:00 CST               8200.0               8200.0
4           adservice                         runtime.jvm_classes_loaded         0.0            31             31    2022-03-21 15:30:00 CST               5162.0               5162.0
5       cartservice-2  mesh.destination.checkoutservice.cartservice.i...         0.0            31             31    2022-03-21 15:30:00 CST               5125.0               5125.0
6   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...         0.0            31             31    2022-03-21 15:30:00 CST               4300.0               4300.0
7      cartservice2-0  mesh.destination.frontend2.cartservice2.istio_...         0.0            31             31    2022-03-21 15:30:00 CST               4100.0               4100.0
8         frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0            31             31    2022-03-21 15:30:00 CST               3100.0               3100.0
9          frontend-2  mesh.destination.unknown.frontend.istio_reques...         0.0            31             31    2022-03-21 15:30:00 CST               2927.5               2927.5
10      cartservice-1  mesh.destination.checkoutservice.cartservice.i...         0.0            31             31    2022-03-21 15:30:00 CST               2620.0               2620.0
11  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...         0.0            31             31    2022-03-21 15:30:00 CST               2150.0               2150.0
12        frontend2-0  mesh.source.frontend2.cartservice2.istio_respo...         0.0            31             31    2022-03-21 15:30:00 CST               1850.0               1850.0
13         frontend-0  mesh.source.frontend.checkoutservice.istio_req...         0.0            31             31    2022-03-21 15:30:00 CST               1650.0               1650.0
14         frontend-1  mesh.source.frontend.checkoutservice.istio_req...         0.0            31             31    2022-03-21 15:30:00 CST               1650.0               1650.0
15      cartservice-2  mesh.destination.checkoutservice.cartservice.i...         0.0            31             31    2022-03-21 15:30:00 CST               1637.5               1637.5
16  checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...         0.0            31             31    2022-03-21 15:30:00 CST               1570.0               1570.0
17     cartservice2-0  mesh.destination.frontend2.cartservice2.istio_...         0.0            31             31    2022-03-21 15:30:00 CST               1310.0               1310.0
18         frontend-0  mesh.source.frontend.checkoutservice.istio_res...         0.0            31             31    2022-03-21 15:30:00 CST               1050.0               1050.0
19         frontend-1  mesh.source.frontend.checkoutservice.istio_res...         0.0            31             31    2022-03-21 15:30:00 CST               1050.0               1050.0,    total_distinct_cmdb_kpi_pairs_in_window  pairs_with_anomaly_count_gt_0
0                                     5350                           4832)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df and tz if present; otherwise load and set tz
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using entire metric_df (must be before any filtering)
# Reuse global_p95 if already computed in this kernel to save time
try:
    global_p95
except NameError:
    global_p95 = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_P95'})
    )

# 2) Filter metric_df to incident window inclusive (Asia/Shanghai)
start_local = pd.Timestamp('2022-03-21 15:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 16:00:00', tz=tz)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Exclude kpi_name containing substrings (case-insensitive): 'mesh.' or 'istio' or 'trace.' or 'mesh'
exclude_patterns = ['mesh\\.', 'istio', 'trace\\.', 'mesh']
exclude_regex = '(' + '|'.join(exclude_patterns) + ')'
mask_exclude = metric_window['kpi_name'].str.contains(exclude_regex, case=False, na=False, regex=True)
metric_window_filtered = metric_window[~mask_exclude].copy()

# 4) Further restrict to KPI names that contain any resource-related keywords (case-insensitive)
keywords = [
    'cpu','memory','rss','heap','jvm','load','transmit','receive','retransmit',
    'packet','latency','drop','disk','io','read','write','space','network','error','process'
]
include_regex = '(' + '|'.join([kw for kw in keywords]) + ')'
mask_include = metric_window_filtered['kpi_name'].str.contains(include_regex, case=False, na=False, regex=True)
metric_window_filtered = metric_window_filtered[mask_include].copy()

# If no rows remain, return empty compact results
if metric_window_filtered.empty:
    top20_filtered = metric_window_filtered.head(0)
    summary_filtered = pd.DataFrame([{
        'distinct_cmdb_kpi_pairs_after_filter': 0,
        'pairs_with_anomaly_count_gt_0': 0
    }])
else:
    # 5) For each remaining (cmdb_id, kpi_name) present in the window compute aggregates
    # Join global_p95 (precomputed) into the windowed filtered data
    dfw = metric_window_filtered.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

    # mark anomalies where value >= global_P95
    dfw['is_anomaly'] = (dfw['value'] >= dfw['global_P95']).astype(int)

    # aggregate window metrics
    agg = (
        dfw
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            window_count=('value', 'size'),
            anomaly_count=('is_anomaly', 'sum'),
            max_value_in_window=('value', 'max')
        )
    )

    # earliest anomaly timestamp per group (if any)
    earliest_anom = (
        dfw[dfw['is_anomaly'] == 1]
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp': 'earliest_anomaly_timestamp_ts'})
    )

    # merge
    res = agg.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left').merge(
        earliest_anom, on=['cmdb_id', 'kpi_name'], how='left'
    )

    # convert earliest timestamp to Asia/Shanghai or None
    def fmt_ts(ts):
        if pd.isna(ts):
            return None
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    res['earliest_anomaly_timestamp'] = res['earliest_anomaly_timestamp_ts'].apply(fmt_ts)
    res.drop(columns=['earliest_anomaly_timestamp_ts'], inplace=True)

    # compute severity and percent_over
    res['max_value_minus_P95'] = res['max_value_in_window'] - res['global_P95']
    # percent_over = (max - p95)/p95 if p95>0 else None
    res['percent_over'] = np.where(res['global_P95'] > 0,
                                   (res['max_value_in_window'] - res['global_P95']) / res['global_P95'],
                                   np.nan)

    # round numeric columns for compactness
    res['global_P95'] = res['global_P95'].round(6)
    res['max_value_in_window'] = res['max_value_in_window'].round(6)
    res['max_value_minus_P95'] = res['max_value_minus_P95'].round(6)
    res['percent_over'] = res['percent_over'].round(6)

    # 6) Order by anomaly_count desc then max_value_minus_P95 desc and limit top 20
    res_sorted = res.sort_values(by=['anomaly_count', 'max_value_minus_P95'], ascending=[False, False])
    out_cols = [
        'cmdb_id','kpi_name','global_P95','window_count','anomaly_count',
        'earliest_anomaly_timestamp','max_value_in_window','max_value_minus_P95','percent_over'
    ]
    top20_filtered = res_sorted[out_cols].head(20).reset_index(drop=True)

    # 7) small aggregates
    total_pairs_filtered = int(res.shape[0])
    pairs_with_anomaly = int((res['anomaly_count'] > 0).sum())
    summary_filtered = pd.DataFrame([{
        'distinct_cmdb_kpi_pairs_after_filter': total_pairs_filtered,
        'pairs_with_anomaly_count_gt_0': pairs_with_anomaly
    }])

# Return compact results: top20_filtered table and summary_filtered single-line aggregate
top20_filtered, summary_filtered

```
Out[3]:
```
```python
# Build a plain-English summary string using existing results (top20_filtered, summary_filtered)
top = top20_filtered
agg = summary_filtered.loc[0]

total_pairs = int(agg['distinct_cmdb_kpi_pairs_after_filter'])
pairs_with_anom = int(agg['pairs_with_anomaly_count_gt_0'])
percent_anom = pairs_with_anom / total_pairs * 100 if total_pairs else 0.0

def fmt_row(r):
    pct = r['percent_over']
    pct_str = f"{pct:.4f}" if pd.notna(pct) else "null"
    return (f"- {r['cmdb_id']} | {r['kpi_name']} -> global_P95={r['global_P95']}, "
            f"window_count={int(r['window_count'])}, anomaly_count={int(r['anomaly_count'])}, "
            f"earliest_anomaly={r['earliest_anomaly_timestamp']}, "
            f"max_in_window={r['max_value_in_window']}, "
            f"max_minus_P95={r['max_value_minus_P95']}, percent_over={pct_str}")

summary_lines = [
    "Resource-focused metric anomaly summary (window: 2022-03-21 15:30:00 to 16:00:00 Asia/Shanghai):",
    f"- Distinct (cmdb_id, kpi_name) pairs after filtering: {total_pairs}",
    f"- Pairs with anomaly_count > 0: {pairs_with_anom} ({percent_anom:.1f}% of filtered pairs)",
    "",
    "Top results (ordered by anomaly_count then severity) — showing up to 20 rows; examples:",
]

# include up to first 5 top rows for briefness (but main output contains top20 in top variable)
n_examples = min(5, len(top))
for i in range(n_examples):
    summary_lines.append(f"{i+1}. {fmt_row(top.iloc[i])}")

summary_lines.append("")
summary_lines.append(
    "Notes: Many top KPIs have anomaly_count = window_count = 31 (every point in the window >= global P95). "
    "Several KPIs have global_P95 = 0, so percent_over is null for those cases."
)

summary_text = "\n".join(summary_lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  global_P95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  max_value_minus_P95  percent_over
0           frontend-2              container.node-5.container_memory_rss  18079744.0            31             31    2022-03-21 15:30:00 CST         1.814733e+07          67584.00000      0.003738
1            adservice                         runtime.jvm_classes_loaded         0.0            31             31    2022-03-21 15:30:00 CST         5.162000e+03           5162.00000           NaN
2   checkoutservice2-0  container.node-6.container_memory_failures.con...         0.0            31             31    2022-03-21 15:30:00 CST         9.900000e+01             99.00000           NaN
3   checkoutservice2-0  container.node-6.container_memory_failures.hie...         0.0            31             31    2022-03-21 15:30:00 CST         9.900000e+01             99.00000           NaN
4               node-6                                    system.io.rkb_s         0.0            31             31    2022-03-21 15:30:00 CST         4.400000e+01             44.00000           NaN
5    checkoutservice-2  container.node-5.container_memory_failures.con...         0.0            31             31    2022-03-21 15:30:00 CST         3.300000e+01             33.00000           NaN
6    checkoutservice-2  container.node-5.container_memory_failures.hie...         0.0            31             31    2022-03-21 15:30:00 CST         3.300000e+01             33.00000           NaN
7          frontend2-0  container.node-6.container_memory_failures.con...         0.0            31             31    2022-03-21 15:30:00 CST         3.300000e+01             33.00000           NaN
8          frontend2-0  container.node-6.container_memory_failures.hie...         0.0            31             31    2022-03-21 15:30:00 CST         3.300000e+01             33.00000           NaN
9   shippingservice2-0  container.node-6.container_memory_failures.con...         0.0            31             31    2022-03-21 15:30:00 CST         3.300000e+01             33.00000           NaN
10  shippingservice2-0  container.node-6.container_memory_failures.hie...         0.0            31             31    2022-03-21 15:30:00 CST         3.300000e+01             33.00000           NaN
11      cartservice2-0       container.node-5.container_fs_reads./dev/vda         0.0            31             31    2022-03-21 15:30:00 CST         2.300000e+01             23.00000           NaN
12   shippingservice-1  container.node-6.container_memory_failures.con...         0.0            31             31    2022-03-21 15:30:00 CST         1.650000e+01             16.50000           NaN
13   shippingservice-1  container.node-6.container_memory_failures.hie...         0.0            31             31    2022-03-21 15:30:00 CST         1.650000e+01             16.50000           NaN
14  shippingservice2-0       container.node-6.container_fs_reads./dev/vda         0.0            31             31    2022-03-21 15:30:00 CST         1.300000e+01             13.00000           NaN
15   checkoutservice-1       container.node-6.container_fs_reads./dev/vda         0.0            31             31    2022-03-21 15:30:00 CST         7.000000e+00              7.00000           NaN
16              node-6                                      system.io.r_s         0.0            31             31    2022-03-21 15:30:00 CST         3.500000e+00              3.50000           NaN
17      cartservice2-0    container.node-5.container_fs_reads_MB./dev/vda         0.0            31             31    2022-03-21 15:30:00 CST         1.093750e+00              1.09375           NaN
18   checkoutservice-0       container.node-6.container_fs_reads./dev/vda         0.0            31             31    2022-03-21 15:30:00 CST         1.000000e+00              1.00000           NaN
19              node-4                          system.process.zombie.num         0.0            31             31    2022-03-21 15:30:00 CST         1.000000e+00              1.00000           NaN,    distinct_cmdb_kpi_pairs_after_filter  pairs_with_anomaly_count_gt_0
0                                  2870                           2701)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

tz = pytz.timezone('Asia/Shanghai')

# 1) Recompute global P95 per (cmdb_id, kpi_name) using entire metric.csv
global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) Filter metric.csv to incident window inclusive (Asia/Shanghai)
start_local = pd.Timestamp('2022-03-21 15:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 16:00:00', tz=tz)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Exclude KPIs containing 'mesh' or 'istio' or 'trace' (case-insensitive)
exclude_regex = r'(mesh\.|istio|trace\.|mesh)'
mask_exclude = metric_window['kpi_name'].str.contains(exclude_regex, case=False, na=False, regex=True)
metric_window = metric_window[~mask_exclude].copy()

# 4) Keep only KPIs containing resource-related keywords (case-insensitive)
keywords = [
    'cpu','memory','rss','heap','jvm','load','transmit','receive','retransmit',
    'packet','latency','drop','disk','io','read','write','space','network','error','process'
]
include_regex = '(' + '|'.join(keywords) + ')'
mask_include = metric_window['kpi_name'].str.contains(include_regex, case=False, na=False, regex=True)
metric_window = metric_window[mask_include].copy()

# If no data remains after filtering, prepare empty outputs
if metric_window.empty:
    top10_cmdb_summary = pd.DataFrame(columns=[
        'cmdb_id','distinct_anomalous_kpis','total_anomaly_points',
        'earliest_anomaly_timestamp','anomalous_kpis_list'
    ]).head(10)
    cmdbs_with_anom_count = pd.DataFrame([{'cmdb_ids_with_anomaly': 0}])
else:
    # Merge global P95 into windowed filtered data (thresholds computed globally)
    dfw = metric_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

    # 5) Mark anomalies where value >= global_P95 (only if global_P95 not NaN)
    dfw['is_anomaly'] = (dfw['global_P95'].notna()) & (dfw['value'] >= dfw['global_P95'])

    # Select anomaly rows
    df_anom = dfw[dfw['is_anomaly']].copy()

    # If no anomalies, return empty results with aggregate zero
    if df_anom.empty:
        top10_cmdb_summary = pd.DataFrame(columns=[
            'cmdb_id','distinct_anomalous_kpis','total_anomaly_points',
            'earliest_anomaly_timestamp','anomalous_kpis_list'
        ]).head(10)
        cmdbs_with_anom_count = pd.DataFrame([{'cmdb_ids_with_anomaly': 0}])
    else:
        # (a) number of distinct kpi_name with anomaly_count>0 per cmdb_id
        distinct_kpis = df_anom.groupby('cmdb_id')['kpi_name'].nunique().rename('distinct_anomalous_kpis')

        # (b) total anomaly data point count across all its KPIs in-window
        total_anom_points = df_anom.groupby('cmdb_id').size().rename('total_anomaly_points')

        # (c) earliest anomaly timestamp among its anomalies (Asia/Shanghai)
        earliest_ts = df_anom.groupby('cmdb_id')['timestamp'].min().rename('earliest_anomaly_ts')

        # (d) list up to 10 distinct kpi_name that had anomalies for that cmdb_id
        kpi_lists = (
            df_anom.groupby('cmdb_id')['kpi_name']
            .unique()
            .apply(lambda arr: list(arr)[:10])
            .rename('anomalous_kpis_list_raw')
        )

        # combine into one DataFrame
        cmdb_summary = pd.concat([distinct_kpis, total_anom_points, earliest_ts, kpi_lists], axis=1).reset_index()

        # format earliest timestamp to Asia/Shanghai string
        def fmt_ts(ts):
            return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        cmdb_summary['earliest_anomaly_timestamp'] = cmdb_summary['earliest_anomaly_ts'].apply(fmt_ts)

        # convert kpi list to comma-separated string (limit to 10 already)
        cmdb_summary['anomalous_kpis_list'] = cmdb_summary['anomalous_kpis_list_raw'].apply(lambda lst: ", ".join(lst))

        # select and order columns
        top_cols = [
            'cmdb_id','distinct_anomalous_kpis','total_anomaly_points',
            'earliest_anomaly_timestamp','anomalous_kpis_list'
        ]
        cmdb_summary = cmdb_summary[top_cols]

        # order by number of distinct anomalous KPIs desc, then by total anomaly points desc
        cmdb_summary_sorted = cmdb_summary.sort_values(
            by=['distinct_anomalous_kpis','total_anomaly_points'],
            ascending=[False, False]
        ).reset_index(drop=True)

        # top 10 cmdb_id
        top10_cmdb_summary = cmdb_summary_sorted.head(10)

        # aggregate: total number of cmdb_id with at least one anomalous KPI
        cmdbs_with_anom_count = pd.DataFrame([{'cmdb_ids_with_anomaly': int(cmdb_summary.shape[0])}])

# Display compact outputs: top10 table and the single-line aggregate
top10_cmdb_summary, cmdbs_with_anom_count

```
Out[4]:
```
```python
# Build a plain-English summary string from the previously computed results
try:
    top = top10_cmdb_summary
    agg = cmdbs_with_anom_count.loc[0]['cmdb_ids_with_anomaly']
except Exception:
    # fallback if variables not present
    import pandas as pd
    top = pd.DataFrame()
    agg = 0

lines = []
lines.append(f"Summary of anomalous cmdb_ids after filtering (resource KPIs, window 2022-03-21 15:30:00 to 16:00:00 Asia/Shanghai):")
lines.append(f"- Total cmdb_id with at least one anomalous KPI: {int(agg)}")
lines.append("")
lines.append("Top 10 cmdb_id by number of distinct anomalous KPIs:")
if top.empty:
    lines.append("- (no anomalous cmdb_id found)")
else:
    for i, r in top.iterrows():
        idx = i + 1
        cmdb = r['cmdb_id']
        distinct = int(r['distinct_anomalous_kpis'])
        points = int(r['total_anomaly_points'])
        earliest = r['earliest_anomaly_timestamp']
        # keep KPI list short for summary (already limited to 10)
        kpis_preview = r['anomalous_kpis_list']
        lines.append(f"{idx}) {cmdb} — {distinct} distinct anomalous KPIs, {points} anomaly points, earliest anomaly {earliest}.")
        # add a short note about KPI examples
        lines.append(f"   Example anomalous KPIs (up to 10): {kpis_preview}")
lines.append("")
lines.append("Note: Many top cmdb_ids show dozens–hundreds of distinct KPIs with anomalies and earliest anomalies at 2022-03-21 15:30:00 CST. KPI name lists are available per cmdb_id if you need the full details.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  distinct_anomalous_kpis  total_anomaly_points earliest_anomaly_timestamp                                anomalous_kpis_list
0          adservice                      319                  8889    2022-03-21 15:30:00 CST  runtime.java_lang_ClassLoading_LoadedClassCoun...
1         adservice2                      317                  8613    2022-03-21 15:30:00 CST  runtime.java_lang_ClassLoading_LoadedClassCoun...
2  shippingservice-1                       51                  1313    2022-03-21 15:30:00 CST  container.node-6.container_cpu_cfs_throttled_p...
3   paymentservice-2                       51                  1302    2022-03-21 15:30:00 CST  container.node-6.container_cpu_cfs_throttled_p...
4         frontend-2                       51                  1272    2022-03-21 15:30:00 CST  container.node-5.container_cpu_cfs_throttled_p...
5  checkoutservice-2                       51                  1239    2022-03-21 15:30:00 CST  container.node-5.container_cpu_load_average_10...
6         frontend-1                       51                  1213    2022-03-21 15:30:00 CST  container.node-5.container_cpu_cfs_throttled_p...
7  shippingservice-2                       51                  1211    2022-03-21 15:30:00 CST  container.node-5.container_cpu_cfs_throttled_p...
8      cartservice-1                       51                  1138    2022-03-21 15:30:00 CST  container.node-6.container_cpu_cfs_throttled_p...
9      cartservice-2                       51                  1071    2022-03-21 15:30:00 CST  container.node-6.container_cpu_cfs_periods, co...,    cmdb_ids_with_anomaly
0                     50)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# reuse trace_df if present otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')

# 1) Global P95 per (cmdb_id, trace_name) using entire trace.csv (must be before filtering)
try:
    trace_global_p95
except NameError:
    trace_global_p95 = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_P95'})
    )

# 2) Filter trace.csv to incident window inclusive (Asia/Shanghai)
start_local = pd.Timestamp('2022-03-21 15:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 16:00:00', tz=tz)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge global P95 into window and define anomaly rule:
# anomaly if value >= global_P95; if global_P95 == 0 then anomaly if value > 0
tw = trace_window.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')

# handle missing global_P95: treat NaN as no threshold -> then anomaly only if value>0? But follow rule: we require comparison;
# we'll treat NaN global_P95 as not anomalous (is_anomaly False) to avoid false positives.
tw['global_P95_filled'] = tw['global_P95'].fillna(np.nan)
tw['is_anomaly'] = False
# cases where global_P95 > 0
mask_gt0 = tw['global_P95_filled'] > 0
tw.loc[mask_gt0, 'is_anomaly'] = (tw.loc[mask_gt0, 'value'] >= tw.loc[mask_gt0, 'global_P95_filled'])
# cases where global_P95 == 0
mask_eq0 = tw['global_P95_filled'] == 0
tw.loc[mask_eq0, 'is_anomaly'] = (tw.loc[mask_eq0, 'value'] > 0)
# remaining NaN global_P95 -> leave False

# 4) For each cmdb_id present in the window compute required aggregates based on anomalies
df_anom = tw[tw['is_anomaly']].copy()

if df_anom.empty:
    top20_cmdb = pd.DataFrame(columns=[
        'cmdb_id','distinct_anomalous_traces','total_anomaly_points',
        'earliest_anomaly_timestamp','example_trace_names_list'
    ]).head(20)
    cmdb_with_anom_agg = pd.DataFrame([{'cmdb_ids_with_anomaly': 0}])
else:
    # (a) number of distinct trace_name with anomaly_count>0 per cmdb_id
    distinct_traces = df_anom.groupby('cmdb_id')['trace_name'].nunique().rename('distinct_anomalous_traces')
    # (b) total anomaly data point count across all those trace_names in-window
    total_points = df_anom.groupby('cmdb_id').size().rename('total_anomaly_points')
    # (c) earliest anomaly timestamp among its anomalies
    earliest_ts = df_anom.groupby('cmdb_id')['timestamp'].min().rename('earliest_anomaly_ts')
    # (d) up to 10 example anomalous trace_name values
    example_traces = (
        df_anom.groupby('cmdb_id')['trace_name']
        .unique()
        .apply(lambda arr: list(arr)[:10])
        .rename('example_trace_names_raw')
    )

    cmdb_summary = pd.concat([distinct_traces, total_points, earliest_ts, example_traces], axis=1).reset_index()

    # format earliest timestamp to Asia/Shanghai
    def fmt_ts(ts):
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    cmdb_summary['earliest_anomaly_timestamp'] = cmdb_summary['earliest_anomaly_ts'].apply(fmt_ts)

    # convert example trace list to comma-separated string
    cmdb_summary['example_trace_names_list'] = cmdb_summary['example_trace_names_raw'].apply(lambda lst: ", ".join(lst))

    out_cols = [
        'cmdb_id','distinct_anomalous_traces','total_anomaly_points',
        'earliest_anomaly_timestamp','example_trace_names_list'
    ]
    cmdb_summary = cmdb_summary[out_cols]

    # 5) Order by distinct_anomalous_traces desc then total_anomaly_points desc and limit top 20
    top20_cmdb = cmdb_summary.sort_values(by=['distinct_anomalous_traces','total_anomaly_points'], ascending=[False, False]).head(20).reset_index(drop=True)

    # 6) single-line aggregate: total number of cmdb_id with at least one anomalous trace
    cmdb_with_anom_agg = pd.DataFrame([{'cmdb_ids_with_anomaly': int(cmdb_summary.shape[0])}])

# Return the compact table and the single-line aggregate
top20_cmdb, cmdb_with_anom_agg

```
Out[5]:
```
```python
# Build a plain-English summary from the trace analysis results (top20_cmdb and cmdb_with_anom_agg)
try:
    tbl = top20_cmdb
    total_cmdbs = int(cmdb_with_anom_agg.loc[0, 'cmdb_ids_with_anomaly'])
except Exception:
    tbl = None
    total_cmdbs = 0

if tbl is None or tbl.empty:
    summary = "No anomalous traces found in the incident window after applying the anomaly rule."
else:
    lines = []
    lines.append("Trace-based localization summary (window: 2022-03-21 15:30:00 to 16:00:00 Asia/Shanghai):")
    lines.append(f"- Total cmdb_id with ≥1 anomalous trace: {total_cmdbs}")
    lines.append("")
    lines.append("Top cmdb_id by distinct anomalous trace names (showing top entries):")

    # show up to first 8 entries for concise summary
    n_show = min(8, len(tbl))
    for i in range(n_show):
        r = tbl.iloc[i]
        lines.append(
            f"{i+1}) {r['cmdb_id']}: {int(r['distinct_anomalous_traces'])} distinct anomalous traces, "
            f"{int(r['total_anomaly_points'])} total anomaly points, earliest anomaly at {r['earliest_anomaly_timestamp']}. "
            f"Examples: {r['example_trace_names_list']}"
        )

    lines.append("")
    lines.append("Key takeaway:")
    lines.append("- The highest counts of distinct anomalous traces appear on checkoutservice-* and frontend-* instances (multiple frontend and checkout cmdb_ids show ~67–69 distinct anomalous trace names).")
    lines.append("- Earliest anomalies for most top cmdb_ids are at 2022-03-21 15:30:00 CST, indicating the issue began at the start of the incident window for those services.")
    lines.append("- Several other services (productcatalogservice, cartservice, recommendationservice, shippingservice) also show anomalies but with fewer distinct trace names and fewer anomaly points.")
    lines.append("")
    lines.append("Interpretation hint: The trace anomalies concentrate on frontend and checkout services (and related call paths between them and downstream services), so those services are the primary candidates for investigation.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  distinct_anomalous_traces  total_anomaly_points earliest_anomaly_timestamp                           example_trace_names_list
0         checkoutservice-1                         69                   220    2022-03-21 15:30:00 CST  trace.from_frontend-0.row_count, trace.to_cart...
1                frontend-1                         69                   213    2022-03-21 15:30:00 CST  trace.to_adservice-2.duration_mean, trace.to_c...
2         checkoutservice-2                         69                   208    2022-03-21 15:30:00 CST  trace.from_frontend-0.row_count, trace.to_curr...
3                frontend-2                         69                   204    2022-03-21 15:30:00 CST  trace.to_cartservice-2.duration_p95, trace.to_...
4                frontend-0                         69                   199    2022-03-21 15:30:00 CST  trace.to_checkoutservice-1.row_count, trace.to...
5         checkoutservice-0                         67                   269    2022-03-21 15:30:00 CST  trace.from_frontend-2.row_count, trace.to_emai...
6   productcatalogservice-2                         30                    84    2022-03-21 15:30:00 CST  trace.from_checkoutservice-1.row_count, trace....
7   productcatalogservice-1                         30                    80    2022-03-21 15:30:00 CST  trace.from_recommendationservice-0.duration_me...
8   productcatalogservice-0                         30                    75    2022-03-21 15:30:00 CST  trace.from_checkoutservice-0.row_count, trace....
9        checkoutservice2-0                         27                    39    2022-03-21 15:49:00 CST  trace.from_frontend2-0.duration_mean, trace.fr...
10              frontend2-0                         27                    29    2022-03-21 15:47:00 CST  trace.from_root.duration_mean, trace.self.dura...
11            cartservice-1                         24                    99    2022-03-21 15:30:00 CST  trace.from_frontend-1.duration_p95, trace.from...
12            cartservice-2                         24                    73    2022-03-21 15:30:00 CST  trace.from_checkoutservice-1.row_count, trace....
13            cartservice-0                         22                    86    2022-03-21 15:30:00 CST  trace.from_checkoutservice-1.row_count, trace....
14  recommendationservice-0                         21                    62    2022-03-21 15:30:00 CST  trace.from_frontend-1.duration_mean, trace.to_...
15  recommendationservice-1                         21                    55    2022-03-21 15:30:00 CST  trace.to_productcatalogservice-1.duration_mean...
16  recommendationservice-2                         21                    50    2022-03-21 15:30:00 CST  trace.to_productcatalogservice-0.duration_mean...
17        shippingservice-0                         18                    55    2022-03-21 15:30:00 CST  trace.from_checkoutservice-1.duration_p95, tra...
18        shippingservice-1                         18                    51    2022-03-21 15:30:00 CST  trace.from_checkoutservice-2.duration_mean, tr...
19        shippingservice-2                         18                    46    2022-03-21 15:30:00 CST  trace.from_checkoutservice-1.duration_mean, tr...,    cmdb_ids_with_anomaly
0                     40)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# reuse existing dataframes if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

tz = pytz.timezone('Asia/Shanghai')

# ----------------------------
# Part 1: Error logs summary
# ----------------------------
# incident window (Asia/Shanghai) inclusive
start_local = pd.Timestamp('2022-03-21 15:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 16:00:00', tz=tz)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# filter error logs to window
err_win = error_logs_df[
    (error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] <= end_ts)
].copy()

if not err_win.empty:
    # group by cmdb_id: count, earliest timestamp, up to 5 example messages
    def earliest_ts_fmt(s):
        return pd.to_datetime(int(s.min()), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    def example_msgs(series):
        # take unique messages in order of appearance, limit to 5, join compactly
        vals = series.dropna().astype(str)
        # preserve order: use groupby cumcount trick
        # but simpler: drop_duplicates keeps first occurrence
        uniq = vals.drop_duplicates().tolist()[:5]
        return " | ".join(uniq)
    error_grp = (
        err_win
        .groupby('cmdb_id', as_index=False)
        .agg(
            error_count=('message', 'size'),
            earliest_ts=('timestamp', 'min'),
            example_messages=('message', lambda s: example_msgs(s))
        )
    )
    error_grp['earliest_error_timestamp'] = error_grp['earliest_ts'].apply(lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'))
    error_summary_df = error_grp.sort_values(by='error_count', ascending=False).reset_index(drop=True)
    # keep compact columns and limit to top 20
    error_summary_df = error_summary_df[['cmdb_id','error_count','earliest_error_timestamp','example_messages']].head(20)
else:
    # empty result
    error_summary_df = pd.DataFrame(columns=['cmdb_id','error_count','earliest_error_timestamp','example_messages'])

# ----------------------------
# Part 2: Node-focused metric anomalies for node-6 and node-5
# ----------------------------
# 1) compute global P95 per (cmdb_id, kpi_name) using entire metric_df (must be before filtering)
try:
    global_p95
except NameError:
    global_p95 = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_P95'})
    )

# 2) filter metric_df to the incident window inclusive
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# focus only on node-6 and node-5
nodes_of_interest = ['node-6','node-5']
metric_nodes = metric_window[metric_window['cmdb_id'].isin(nodes_of_interest)].copy()

# if no data, prepare empty outputs
node_tables = {}
node_aggregates = []

if metric_nodes.empty:
    for n in nodes_of_interest:
        node_tables[n] = pd.DataFrame(columns=[
            'node_cmdb_id','kpi_name','global_P95','window_count','anomaly_count',
            'earliest_anomaly_timestamp','max_value_in_window','max_value_minus_P95'
        ])
        node_aggregates.append({
            'node_cmdb_id': n,
            'distinct_kpi_with_anomaly': 0,
            'earliest_anomaly_timestamp_across_kpis': None
        })
else:
    # merge global P95 thresholds into the node window data
    mn = metric_nodes.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

    # define anomaly per rules:
    # anomaly if global_P95 > 0 and value >= global_P95
    # if global_P95 == 0 then anomaly if value > 0
    # if global_P95 is NaN -> treat as no anomaly (False)
    mn['global_P95_filled'] = mn['global_P95']  # may contain NaN
    mn['is_anomaly'] = False
    mask_p95_gt0 = mn['global_P95_filled'] > 0
    mn.loc[mask_p95_gt0, 'is_anomaly'] = mn.loc[mask_p95_gt0, 'value'] >= mn.loc[mask_p95_gt0, 'global_P95_filled']
    mask_p95_eq0 = mn['global_P95_filled'] == 0
    mn.loc[mask_p95_eq0, 'is_anomaly'] = mn.loc[mask_p95_eq0, 'value'] > 0
    # NaN stays False

    # aggregate per node and kpi_name
    grp = (
        mn
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            window_count=('value','size'),
            anomaly_count=('is_anomaly','sum'),
            max_value_in_window=('value','max')
        )
    )

    # earliest anomaly timestamp per (node,kpi) where anomaly_count>0
    earliest_anom = (
        mn[mn['is_anomaly']]
        .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anom_ts'})
    )

    # merge earliest and global_p95 into grp
    grp = grp.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')
    grp = grp.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

    # compute max_value_minus_P95
    grp['max_value_minus_P95'] = grp['max_value_in_window'] - grp['global_P95']

    # format earliest timestamp to Asia/Shanghai or None
    def fmt_ts_nullable(ts):
        if pd.isna(ts):
            return None
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    grp['earliest_anomaly_timestamp'] = grp['earliest_anom_ts'].apply(fmt_ts_nullable)
    grp.drop(columns=['earliest_anom_ts'], inplace=True)

    # round numeric columns for compactness
    grp['global_P95'] = grp['global_P95'].round(6)
    grp['max_value_in_window'] = grp['max_value_in_window'].round(6)
    grp['max_value_minus_P95'] = grp['max_value_minus_P95'].round(6)

    # For each node, filter to anomalous KPIs (anomaly_count>0), order and limit top 20
    for n in nodes_of_interest:
        node_grp = grp[grp['cmdb_id'] == n].copy()
        node_anom = node_grp[node_grp['anomaly_count'] > 0].copy()
        if node_anom.empty:
            node_table = pd.DataFrame(columns=[
                'node_cmdb_id','kpi_name','global_P95','window_count','anomaly_count',
                'earliest_anomaly_timestamp','max_value_in_window','max_value_minus_P95'
            ])
            distinct_kpi_count = 0
            earliest_overall = None
        else:
            node_anom['node_cmdb_id'] = n
            # order by anomaly_count desc then max_value_minus_P95 desc
            node_anom_sorted = node_anom.sort_values(by=['anomaly_count','max_value_minus_P95'], ascending=[False, False])
            node_table = node_anom_sorted[[
                'node_cmdb_id','kpi_name','global_P95','window_count','anomaly_count',
                'earliest_anomaly_timestamp','max_value_in_window','max_value_minus_P95'
            ]].head(20).reset_index(drop=True)
            distinct_kpi_count = int(node_anom['kpi_name'].nunique())
            # earliest anomaly timestamp across all KPIs for this node
            ts_min = node_anom['earliest_anomaly_timestamp'].dropna().min()
            earliest_overall = ts_min if pd.notna(ts_min) else None

        node_tables[n] = node_table
        node_aggregates.append({
            'node_cmdb_id': n,
            'distinct_kpi_with_anomaly': distinct_kpi_count,
            'earliest_anomaly_timestamp_across_kpis': earliest_overall
        })

# assemble aggregates DataFrame
node_aggregates_df = pd.DataFrame(node_aggregates)

# Keep outputs compact: top 20 error_summary_df, node tables (node-6 and node-5) and node aggregates
# Final displayed variables:
error_summary_df, node_tables.get('node-6'), node_tables.get('node-5'), node_aggregates_df

```
Out[6]:
```
Summary of findings (incident window 2022-03-21 15:30:00 — 16:00:00 Asia/Shanghai):

1) Error logs (top cmdb_id by error count)
- frontend-1: 204 error rows, earliest error 2022-03-21 15:45:42 CST. Example messages include severity: error, message: request error, and related GET failures.
- frontend-2: 146 error rows, earliest 2022-03-21 15:45:39 CST. Similar request error messages.
- cartservice-1: 88 error rows, earliest 2022-03-21 15:45:54 CST. Example messages include "Error status code 'FailedPrecondition'".
- cartservice-2: 84 error rows, earliest 2022-03-21 15:45:39 CST.
- frontend-0: 70 error rows, earliest 2022-03-21 15:45:41 CST.
- cartservice-0: 48 error rows, earliest 2022-03-21 15:46:04 CST.
- (Results were produced for up to top 20 cmdb_ids; the pattern shows frontends and cartservices dominate error counts.)

Interpretation: high error volumes are concentrated on frontend instances and cartservice instances, with errors beginning ~15:45–15:46 for many of these hosts.

2) Node-focused metric anomaly summary (node-6 and node-5)
- Method: global P95 computed across the entire metric file; anomalies in-window defined as value >= global_P95 (or value>0 if global_P95==0). Window: 2022-03-21 15:30:00–16:00:00 CST.

Node-6 (summary)
- Distinct kpi_name with anomaly_count > 0: 33
- Earliest anomaly across KPIs: 2022-03-21 15:30:00 CST
- Notable anomalous KPIs (top examples, many had anomaly_count = 31):
  - ping.can_connect (global_P95=1.0) — anomaly_count=31 (every point), earliest 15:30:00
  - system.mem.total, system.os.nofile.max, system.tcp.retrans_pct — anomaly_count=31
  - system.fs.inodes.in_use — anomaly_count=30
  - system.net.tcp.retrans_segs — anomaly_count=4 (earliest 15:46:00)
  - system.io.w_s — anomaly_count=3, max_value_minus_P95 ≈ 109.5 (IO write/s spike)
  - system.disk.used — anomaly_count=1, but very large max_value_minus_P95 ≈ 143,945 (large disk usage jump at 16:00)
  - system.cpu.system, system.cpu.iowait, system.io.w_await — small but visible increases
- Interpretation: node-6 shows many metrics above their historical P95 across the whole window (start at 15:30). There are IO/write and disk usage spikes and some TCP retransmission indications.

Node-5 (summary)
- Distinct kpi_name with anomaly_count > 0: 42
- Earliest anomaly across KPIs: 2022-03-21 15:30:00 CST
- Notable anomalous KPIs (top examples, many anomaly_count = 31):
  - system.udp.connect.num, system.disk.total, system.fs.inodes.in_use, system.mem.total, system.os.nofile.max, system.tcp.retrans_pct — anomaly_count=31
  - system.os.nofile.used_pct — anomaly_count=12 (earliest 15:38)
  - system.os.nofile.current — anomaly_count=6 (earliest 15:38)
  - system.net.udp.out_datagrams — anomaly_count=3, max_value_minus_P95 ≈ 38.98 (UDP/network spike)
  - system.io.w_s — anomaly_count=3, max_value_minus_P95 ≈ 10.5 (IO write/s increase)
  - system.mem.used — anomaly_count=3, max_value_minus_P95 ≈ 90.0
- Interpretation: node-5 shows even more distinct KPIs anomalous than node-6, including UDP/network and IO/disk/memory signals.

Combined interpretation and recommended focus
- Logs indicate frontend and cartservice instances exhibited many errors (frontends highest). Traces earlier analysis pointed to frontend/checkout hotspots as well.
- Node metrics show both node-5 and node-6 had many KPIs exceeding their global P95 starting at the beginning of the window (15:30:00 CST). Node-5 has more distinct anomalous KPIs (42) than node-6 (33).
- Key node-level signals to investigate: disk usage and disk IO/write spikes, increased IO wait, elevated TCP/UDP retransmissions/out datagrams, and memory usage changes — these can affect service requests and produce the observed frontend/cartservice errors.
- Recommended next steps: investigate disk space/IO and network health on node-5 and node-6 (check disk usage, ongoing writes, IO wait, network packet retransmits), correlate with the failing frontends/cartservice pods scheduled on those nodes, and inspect processes/containers on those nodes for heavy IO or network activity.

If you want, I can produce the full per-node anomalous KPI table (top 20 per node) or the full error message excerpts for a specific cmdb_id.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  error_count earliest_error_timestamp                                   example_messages
0     frontend-1          204  2022-03-21 15:45:42 CST  severity: error, message: request error | "GET...
1     frontend-2          146  2022-03-21 15:45:39 CST  severity: error, message: request error | "GET...
2  cartservice-1           88  2022-03-21 15:45:54 CST       Error status code 'FailedPrecondition' ra...
3  cartservice-2           84  2022-03-21 15:45:39 CST       Error status code 'FailedPrecondition' ra...
4     frontend-0           70  2022-03-21 15:45:41 CST  severity: error, message: request error | "GET...
5  cartservice-0           48  2022-03-21 15:46:04 CST       Error status code 'FailedPrecondition' ra...,    node_cmdb_id                      kpi_name    global_P95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  max_value_minus_P95
0        node-6              ping.can_connect  1.000000e+00            31             31    2022-03-21 15:30:00 CST         1.000000e+00                0.000
1        node-6              system.mem.total  3.211700e+04            31             31    2022-03-21 15:30:00 CST         3.211700e+04                0.000
2        node-6          system.os.nofile.max  6.553600e+06            31             31    2022-03-21 15:30:00 CST         6.553600e+06                0.000
3        node-6        system.tcp.retrans_pct  1.000000e-02            31             31    2022-03-21 15:30:00 CST         1.000000e-02                0.000
4        node-6       system.fs.inodes.in_use  1.170000e+00            31             30    2022-03-21 15:30:00 CST         1.170000e+00                0.000
5        node-6   system.net.tcp.retrans_segs  1.510000e+00            31              4    2022-03-21 15:46:00 CST         1.950000e+00                0.440
6        node-6                 system.io.w_s  9.050000e+01            31              3    2022-03-21 15:43:00 CST         2.000000e+02              109.500
7        node-6      system.os.nofile.current  1.286400e+04            31              3    2022-03-21 15:50:00 CST         1.292800e+04               64.000
8        node-6             system.cpu.system  5.290000e+00            31              3    2022-03-21 15:43:00 CST         5.890000e+00                0.600
9        node-6             system.cpu.iowait  8.000000e-02            31              3    2022-03-21 15:43:00 CST         3.000000e-01                0.220
10       node-6             system.io.w_await  9.200000e-01            31              3    2022-03-21 15:31:00 CST         9.900000e-01                0.070
11       node-6            system.io.avg_q_sz  1.000000e-01            31              3    2022-03-21 15:43:00 CST         1.200000e-01                0.020
12       node-6     system.os.nofile.used_pct  1.960000e-01            31              3    2022-03-21 15:50:00 CST         1.970000e-01                0.001
13       node-6   system.net.udp.in_datagrams  9.334000e+02            31              2    2022-03-21 15:55:00 CST         9.451500e+02               11.750
14       node-6  system.net.udp.out_datagrams  1.386500e+02            31              2    2022-03-21 15:55:00 CST         1.460500e+02                7.400
15       node-6                system.io.util  1.850000e+00            31              2    2022-03-21 15:43:00 CST         8.150000e+00                6.300
16       node-6                 system.load.1  2.330000e+00            31              2    2022-03-21 15:38:00 CST         3.570000e+00                1.240
17       node-6               system.io.await  9.300000e-01            31              2    2022-03-21 15:31:00 CST         9.900000e-01                0.060
18       node-6              system.disk.used  6.267670e+09            31              1    2022-03-21 16:00:00 CST         6.267814e+09           143945.140
19       node-6               system.mem.used  6.930000e+03            31              1    2022-03-21 16:00:00 CST         7.523000e+03              593.000,    node_cmdb_id                      kpi_name    global_P95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  max_value_minus_P95
0        node-5        system.udp.connect.num  1.100000e+01            31             31    2022-03-21 15:30:00 CST         1.200000e+01                1.000
1        node-5             system.disk.total  1.854945e+10            31             31    2022-03-21 15:30:00 CST         1.854945e+10                0.000
2        node-5       system.fs.inodes.in_use  4.700000e-01            31             31    2022-03-21 15:30:00 CST         4.700000e-01                0.000
3        node-5              system.mem.total  3.211700e+04            31             31    2022-03-21 15:30:00 CST         3.211700e+04                0.000
4        node-5          system.os.nofile.max  6.553600e+06            31             31    2022-03-21 15:30:00 CST         6.553600e+06                0.000
5        node-5        system.tcp.retrans_pct  1.000000e-02            31             31    2022-03-21 15:30:00 CST         1.000000e-02                0.000
6        node-5     system.os.nofile.used_pct  1.350000e-01            31             12    2022-03-21 15:38:00 CST         1.360000e-01                0.001
7        node-5      system.os.nofile.current  8.864000e+03            31              6    2022-03-21 15:38:00 CST         8.896000e+03               32.000
8        node-5         system.fs.inodes.used  4.635594e+06            31              4    2022-03-21 15:50:00 CST         4.635617e+06               23.000
9        node-5    system.mem.real.pct_useage  4.589000e+01            31              4    2022-03-21 15:54:00 CST         4.618000e+01                0.290
10       node-5   system.net.tcp.retrans_segs  4.300000e-01            31              4    2022-03-21 15:46:00 CST         6.500000e-01                0.220
11       node-5                system.load.15  1.140000e+00            31              4    2022-03-21 15:31:00 CST         1.160000e+00                0.020
12       node-5         system.disk.pct_usage  6.869000e+01            31              4    2022-03-21 15:57:00 CST         6.869000e+01                0.000
13       node-5     system.process.zombie.num  3.500000e+01            31              4    2022-03-21 15:32:00 CST         3.500000e+01                0.000
14       node-5               system.mem.used  1.452900e+04            31              3    2022-03-21 15:54:00 CST         1.461900e+04               90.000
15       node-5  system.net.udp.out_datagrams  4.500000e+00            31              3    2022-03-21 15:36:00 CST         4.348000e+01               38.980
16       node-5                 system.io.w_s  2.650000e+01            31              3    2022-03-21 15:37:00 CST         3.700000e+01               10.500
17       node-5               system.cpu.user  6.270000e+00            31              3    2022-03-21 15:34:00 CST         6.960000e+00                0.690
18       node-5          system.mem.pct_usage  4.524000e+01            31              3    2022-03-21 15:54:00 CST         4.552000e+01                0.280
19       node-5             system.cpu.iowait  4.000000e-02            31              3    2022-03-21 15:47:00 CST         8.000000e-02                0.040,   node_cmdb_id  distinct_kpi_with_anomaly earliest_anomaly_timestamp_across_kpis
0       node-6                         33                2022-03-21 15:30:00 CST
1       node-5                         42                2022-03-21 15:30:00 CST)```
```